In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, f1_score, recall_score, precision_score,
    average_precision_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve, precision_recall_curve
)
import mlflow
import mlflow.pytorch
import mlflow.sklearn
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

C:\Users\Guima\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


In [4]:
# --- Carregar e preparar dados ---
import kagglehub
from kagglehub import KaggleDatasetAdapter

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "blastchar/telco-customer-churn",
    "WA_Fn-UseC_-Telco-Customer-Churn.csv",
)

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors='coerce')
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())
df.rename(columns={'Churn': 'target'}, inplace=True)
df['target'] = df['target'].map({'No': 0, 'Yes': 1})

X = df.drop(columns=['customerID', 'target'])
y = df['target']

numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_features = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaperlessBilling', 'PaymentMethod'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
    ]
)

print(f"Dataset: {X.shape[0]} amostras, {X.shape[1]} features")
print(f"Target: {y.value_counts(normalize=True).to_dict()}")

Dataset: 7043 amostras, 19 features
Target: {0: 0.7346301292063041, 1: 0.2653698707936959}


In [5]:
class ChurnMLP(nn.Module):
    """
    Multi-Layer Perceptron para predição de churn.
    Arquitetura: Input → 128 → 64 → 32 → 1
    Com BatchNorm e Dropout para regularização.
    """
    def __init__(self, input_dim, dropout_rate=0.3):
        super().__init__()
        self.network = nn.Sequential(
            # Camada 1
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            # Camada 2
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            # Camada 3
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            # Output
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.network(x)

print("Arquitetura definida: Input → 128 → 64 → 32 → 1")
print("Ativação: ReLU | Regularização: BatchNorm + Dropout(0.3)")
print("Loss: BCELoss (Binary Cross Entropy)")

Arquitetura definida: Input → 128 → 64 → 32 → 1
Ativação: ReLU | Regularização: BatchNorm + Dropout(0.3)
Loss: BCELoss (Binary Cross Entropy)


## Decisões de Arquitetura

- **3 camadas ocultas (128 → 64 → 32)**: rede compacta para dados tabulares (~7k amostras).
  Redes muito profundas não ajudam em datasets tabulares pequenos.
- **ReLU**: ativação padrão — resolve vanishing gradient, computacionalmente eficiente.
- **BatchNorm**: estabiliza o treinamento normalizando as ativações entre camadas.
- **Dropout (0.3)**: regularização para evitar overfitting — desliga 30% dos neurônios a cada batch.
- **Sigmoid na saída**: classificação binária — output entre 0 e 1 (probabilidade de churn).
- **BCELoss**: loss padrão para classificação binária com saída sigmoid.